# leankg — health_stress (English) demo

Builds a small knowledge graph from the hand-written sample documents with the LLM
**off** (`llm="none"`), so it runs without any API key. Then it runs three Cypher queries
(one in Japanese) if Neo4j is up, or the NetworkX equivalents otherwise.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # so leankg finds domains/ and leankg.toml
from leankg import Pipeline
from leankg.report import render_report

In [ ]:
pipe = Pipeline.from_domain("health_stress", llm="none")
report = pipe.build(ROOT / "examples" / "health_stress", out="json", out_path="out/health_stress")
print(render_report(report))

## Facts extracted by rules (accepted and negative evidence)

In [ ]:
ents = pipe.graph.entities
for t in sorted(pipe.graph.facts(), key=lambda t: -t.confidence):
    neg = "NOT " if t.negated else ""
    print(f"{ents[t.head].name:28} {neg}{t.type:18} {ents[t.tail].name:28} "
          f"{t.method:8} conf={t.confidence:.2f} langs={t.langs}")

## Three queries (Neo4j, or NetworkX fallback)

In [ ]:
# Connect to Neo4j if it's running (docker compose up -d); otherwise use NetworkX.
from leankg.config import load_config

cfg = load_config()
writer = None
try:
    from leankg.export.neo4j_writer import Neo4jWriter
    pw = cfg.neo4j.password.get_secret_value() if cfg.neo4j.password else "leankg-dev-pw"
    writer = Neo4jWriter(cfg.neo4j.uri, cfg.neo4j.user, pw)
    writer.driver.verify_connectivity()
    writer.clear()  # demo database only!
    print(writer.write_graph(pipe.graph))
    writer.run("CALL db.awaitIndexes(60)")
except Exception as exc:  # no server / no [neo4j] extra
    writer = None
    print(f"Neo4j not available ({type(exc).__name__}); falling back to NetworkX")

from leankg.export import to_networkx
G = to_networkx(pipe.graph)

def cypher(query, **params):
    if writer is None:
        return None
    return [dict(r) for r in writer.run(query, **params)]

In [ ]:
# 1. What increases the risk of insomnia?
rows = cypher("MATCH (s)-[r:INCREASES_RISK_OF {negated: false}]->(c {id: 'HS:insomnia'}) "
              "RETURN s.name AS cause, r.evidence_count AS evidence")
if rows is None:
    rows = [{"cause": G.nodes[u]["name"], "evidence": d["evidence_count"]}
            for u, v, k, d in G.edges(keys=True, data=True)
            if v == "HS:insomnia" and k == "INCREASES_RISK_OF" and d.get("negated") is False]
rows

In [ ]:
# 2. Negative evidence: interventions that did NOT reduce something
rows = cypher("MATCH (i:Intervention)-[r:REDUCES {negated: true}]->(x) RETURN i.name AS intervention, x.name AS target")
if rows is None:
    rows = [{"intervention": G.nodes[u]["name"], "target": G.nodes[v]["name"]}
            for u, v, k, d in G.edges(keys=True, data=True) if k == "REDUCES" and d.get("negated") is True]
rows

In [ ]:
# 3. Japanese input: full-text search with the CJK analyzer
rows = cypher("CALL db.index.fulltext.queryNodes('entity_names', $q) YIELD node, score "
              "RETURN node.id AS id, node.name AS name, node.name_ja AS name_ja, score", q="ストレス")
if rows is None:
    rows = [{"id": n, "name": d.get("name"), "name_ja": d.get("name_ja")}
            for n, d in G.nodes(data=True) if "ストレス" in str(d.get("aliases_text", "")) + str(d.get("name_ja", ""))]
rows